<a href="https://colab.research.google.com/github/hari-hara-sudharsan/MLops-Pro/blob/main/ResumeMatch.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### Load Data into a DataFrame

Now that the dataset is downloaded, we can load it into a pandas DataFrame to begin our analysis. We'll assume the main data file is a CSV within the downloaded path.

In [ ]:
import pandas as pd
import os
import kagglehub

# Re-download the dataset to define the 'path' variable
path = kagglehub.dataset_download("snehaanbhawal/resume-dataset")
print(f"Path to dataset files: {path}")

# Assuming the main data file is a CSV in the downloaded path.
# We need to find the CSV file within the downloaded directory.
# Let's list the files first to confirm.

print(f"Files in the dataset directory: {os.listdir(path)}")

# The previous attempt failed because the CSV was likely nested.
# Let's check inside the 'Resume' subdirectory, as it's common for Kaggle datasets.
resume_dir = os.path.join(path, 'Resume')
print(f"Files in 'Resume' subdirectory: {os.listdir(resume_dir)}")

# Now, try to find and load the CSV from the 'Resume' subdirectory.
try:
    csv_file = [f for f in os.listdir(resume_dir) if f.endswith('.csv')][0]
    df = pd.read_csv(os.path.join(resume_dir, csv_file))
    print(f"Successfully loaded {csv_file} from {resume_dir}")
except IndexError:
    print("No CSV file found in the 'Resume' subdirectory. Please check its content.")
except Exception as e:
    print(f"An error occurred while loading the CSV: {e}")

Using Colab cache for faster access to the 'resume-dataset' dataset.
Path to dataset files: /kaggle/input/resume-dataset
Files in the dataset directory: ['Resume', 'data']
Files in 'Resume' subdirectory: ['Resume.csv']
Successfully loaded Resume.csv from /kaggle/input/resume-dataset/Resume


### Display the first 5 rows of the DataFrame

In [ ]:
if 'df' in locals() and not df.empty:
    display(df.head())
else:
    print("DataFrame 'df' was not created or is empty. Please check the previous cell's output for errors.")

,ID,Resume_str,Resume_html,Category
0,16852973,HR ADMINISTRATOR/MARKETING ASSOCIATE\...,"<div class=""fontsize fontface vmargins hmargin...",HR
1,22323967,"HR SPECIALIST, US HR OPERATIONS ...","<div class=""fontsize fontface vmargins hmargin...",HR
2,33176873,HR DIRECTOR Summary Over 2...,"<div class=""fontsize fontface vmargins hmargin...",HR
3,27018550,HR SPECIALIST Summary Dedica...,"<div class=""fontsize fontface vmargins hmargin...",HR
4,17812897,HR MANAGER Skill Highlights ...,"<div class=""fontsize fontface vmargins hmargin...",HR


### Initial Data Exploration

Let's get a summary of the DataFrame to understand its structure, check for missing values, and inspect data types.

In [ ]:
# Display basic info about the DataFrame
display(df.info())

# Check for missing values
print("\nMissing values per column:")
display(df.isnull().sum())

# Display the number of unique categories
print("\nNumber of unique resume categories:", df['Category'].nunique())
print("\nUnique categories:")
display(df['Category'].value_counts())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2484 entries, 0 to 2483
Data columns (total 4 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   ID           2484 non-null   int64 
 1   Resume_str   2484 non-null   object
 2   Resume_html  2484 non-null   object
 3   Category     2484 non-null   object
dtypes: int64(1), object(3)
memory usage: 77.8+ KB


None


Missing values per column:


,0
ID,0
Resume_str,0
Resume_html,0
Category,0



Number of unique resume categories: 24

Unique categories:


,count
Category,
INFORMATION-TECHNOLOGY,120
BUSINESS-DEVELOPMENT,120
ADVOCATE,118
CHEF,118
ENGINEERING,118
ACCOUNTANT,118
FINANCE,118
FITNESS,117
AVIATION,117


In [ ]:
display(df['Category'].value_counts())

,count
Category,
INFORMATION-TECHNOLOGY,120
BUSINESS-DEVELOPMENT,120
ADVOCATE,118
CHEF,118
ENGINEERING,118
ACCOUNTANT,118
FINANCE,118
FITNESS,117
AVIATION,117


### Text Preprocessing

To prepare the resume text for analysis and model training, we need to clean it. This typically involves several steps:

1.  **Remove URLs and Special Characters**: Get rid of web links, punctuation, and other non-alphanumeric characters.
2.  **Convert to Lowercase**: Ensure consistency by converting all text to lowercase.
3.  **Remove Stop Words**: Eliminate common words that don't carry much meaning (e.g., 'the', 'is', 'a').
4.  **Tokenization**: Break down the text into individual words or tokens.
5.  **Stemming or Lemmatization**: Reduce words to their base or root form to handle variations (e.g., 'running', 'runs', 'ran' -> 'run').

Let's start by cleaning URLs and special characters and converting the text to lowercase.

In [ ]:
import re
import string

def clean_text(text):
    # Remove URLs
    text = re.sub('http\S+\s*', '', text)  # remove URLs
    text = re.sub('RT|cc', '', text)  # remove RT and cc
    text = re.sub('#\S+', '', text)  # remove hashtags
    text = re.sub('@\S+', '', text)  # remove mentions
    text = re.sub('[%s]' % re.escape(string.punctuation), '', text)  # remove punctuation
    text = re.sub('[^a-zA-Z]', ' ', text)  # remove non-alphabetic characters
    text = re.sub('\s+', ' ', text)  # remove extra whitespace
    text = text.lower() # convert to lowercase
    return text

df['cleaned_resume'] = df['Resume_str'].apply(lambda x: clean_text(x))

print("Original Resume Sample:\n", df['Resume_str'][0])
print("\nCleaned Resume Sample:\n", df['cleaned_resume'][0])

<>:6: SyntaxWarning: invalid escape sequence '\S'
<>:8: SyntaxWarning: invalid escape sequence '\S'
<>:9: SyntaxWarning: invalid escape sequence '\S'
<>:12: SyntaxWarning: invalid escape sequence '\s'
<>:6: SyntaxWarning: invalid escape sequence '\S'
<>:8: SyntaxWarning: invalid escape sequence '\S'
<>:9: SyntaxWarning: invalid escape sequence '\S'
<>:12: SyntaxWarning: invalid escape sequence '\s'
/tmp/ipykernel_3116/124783332.py:6: SyntaxWarning: invalid escape sequence '\S'
  text = re.sub('http\S+\s*', '', text)  # remove URLs
/tmp/ipykernel_3116/124783332.py:8: SyntaxWarning: invalid escape sequence '\S'
  text = re.sub('#\S+', '', text)  # remove hashtags
/tmp/ipykernel_3116/124783332.py:9: SyntaxWarning: invalid escape sequence '\S'
  text = re.sub('@\S+', '', text)  # remove mentions
/tmp/ipykernel_3116/124783332.py:12: SyntaxWarning: invalid escape sequence '\s'
  text = re.sub('\s+', ' ', text)  # remove extra whitespace


Original Resume Sample:
          HR ADMINISTRATOR/MARKETING ASSOCIATE

HR ADMINISTRATOR       Summary     Dedicated Customer Service Manager with 15+ years of experience in Hospitality and Customer Service Management.   Respected builder and leader of customer-focused teams; strives to instill a shared, enthusiastic commitment to customer service.         Highlights         Focused on customer satisfaction  Team management  Marketing savvy  Conflict resolution techniques     Training and development  Skilled multi-tasker  Client relations specialist           Accomplishments      Missouri DOT Supervisor Training Certification  Certified by IHG in Customer Loyalty and Marketing by Segment   Hilton Worldwide General Manager Training Certification  Accomplished Trainer for cross server hospitality systems such as    Hilton OnQ  ,   Micros    Opera PMS   , Fidelio    OPERA    Reservation System (ORS) ,   Holidex    Completed courses and seminars in customer service, sales strategies, inve

### Stop Word Removal and Tokenization

Next, we'll remove common stop words (like 'the', 'is', 'a') that typically don't carry significant meaning for classification. After that, we'll tokenize the cleaned text, breaking it down into individual words, which is essential for further feature engineering.

In [ ]:
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

# Download NLTK stop words if not already downloaded
try:
    stopwords.words('english')
except LookupError:
    nltk.download('stopwords')
# Download 'punkt_tab' for tokenization, as suggested by the previous error
try:
    word_tokenize('test')
except LookupError:
    nltk.download('punkt_tab') # Changed from 'punkt' to 'punkt_tab'

stop_words = set(stopwords.words('english'))

def remove_stopwords_and_tokenize(text):
    word_tokens = word_tokenize(text)
    filtered_sentence = [w for w in word_tokens if not w in stop_words]
    return ' '.join(filtered_sentence)

df['processed_resume'] = df['cleaned_resume'].apply(remove_stopwords_and_tokenize)

print("Cleaned Resume Sample (before stop word removal and tokenization):\n", df['cleaned_resume'][0])
print("\nProcessed Resume Sample (after stop word removal and tokenization):\n", df['processed_resume'][0])

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


Cleaned Resume Sample (before stop word removal and tokenization):
  hr administratormarketing associate hr administrator summary dedicated customer service manager with years of experience in hospitality and customer service management respected builder and leader of customerfocused teams strives to instill a shared enthusiastic commitment to customer service highlights focused on customer satisfaction team management marketing savvy conflict resolution techniques training and development skilled multitasker client relations specialist aomplishments missouri dot supervisor training certification certified by ihg in customer loyalty and marketing by segment hilton worldwide general manager training certification aomplished trainer for cross server hospitality systems such as hilton onq micros opera pms fidelio opera reservation system ors holidex completed courses and seminars in customer service sales strategies inventory control loss prevention safety time management leadership and p

### Feature Engineering: Text Vectorization (TF-IDF)

To prepare our processed resume text for machine learning models, we need to convert it into a numerical representation. TF-IDF (Term Frequency-Inverse Document Frequency) is a popular technique that reflects how important a word is to a document in a collection or corpus. It increases proportionally to the number of times a word appears in the document but is offset by the frequency of the word in the corpus, which helps to adjust for the fact that some words appear more frequently in general.

We will use `TfidfVectorizer` from `sklearn.feature_extraction.text` to achieve this.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

# Initialize TF-IDF Vectorizer
# You can adjust parameters like max_features, min_df, max_df as needed
tfidf_vectorizer = TfidfVectorizer(stop_words='english', max_features=5000) # Limiting to 5000 features for now

# Fit and transform the 'processed_resume' column
X = tfidf_vectorizer.fit_transform(df['processed_resume'])

# Display the shape of the resulting TF-IDF matrix
print(f"Shape of TF-IDF matrix: {X.shape}")

# Also, let's store the feature names (words) for later interpretability
tfidf_feature_names = tfidf_vectorizer.get_feature_names_out()
print(f"Number of TF-IDF features: {len(tfidf_feature_names)}")

Shape of TF-IDF matrix: (2484, 5000)
Number of TF-IDF features: 5000


### Model Selection & Training

Now that we have our numerical features (`X`) from the TF-IDF vectorization, we need to prepare our target variable (`Category`) and then split the data for training and testing. We'll use `LabelEncoder` to convert the categorical labels into numerical values and `train_test_split` to divide the data.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

# Encode the target variable 'Category'
le = LabelEncoder()
y = le.fit_transform(df['Category'])

# Split the data into training and testing sets
# We'll use 80% for training and 20% for testing
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print(f"Shape of X_train: {X_train.shape}")
print(f"Shape of X_test: {X_test.shape}")
print(f"Shape of y_train: {y_train.shape}")
print(f"Shape of y_test: {y_test.shape}")

print("\nUnique encoded categories in y_train:", len(set(y_train)))
print("Unique encoded categories in y_test:", len(set(y_test)))

Shape of X_train: (1987, 5000)
Shape of X_test: (497, 5000)
Shape of y_train: (1987,)
Shape of y_test: (497,)

Unique encoded categories in y_train: 24
Unique encoded categories in y_test: 24


### Train a Classification Model (Support Vector Machine)

We'll use a Support Vector Machine (SVC) with a linear kernel, which is often a good choice for text classification tasks with high-dimensional data like TF-IDF features. SVC will learn to classify the resumes into the 24 categories.

In [ ]:
from sklearn.svm import SVC
from sklearn.metrics import classification_report, accuracy_score

# Initialize the Support Vector Classifier
# Using a linear kernel for better interpretability and performance on high-dimensional text data
svc_model = SVC(kernel='linear', random_state=42)

# Train the model
print("Training the SVM model...")
svc_model.fit(X_train, y_train)
print("Model training complete.")

# Make predictions on the test set
y_pred = svc_model.predict(X_test)

# Evaluate the model
print("\nModel Evaluation:")
print(f"Accuracy: {accuracy_score(y_test, y_pred):.4f}")

# To see the full classification report, we need the original category names
# Let's inverse transform y_test and y_pred to get original labels for the report

# First, ensure 'le' (LabelEncoder) is available. It was defined in a previous cell.
original_y_test_labels = le.inverse_transform(y_test)
original_y_pred_labels = le.inverse_transform(y_pred)

print("\nClassification Report:")
print(classification_report(original_y_test_labels, original_y_pred_labels))

Training the SVM model...
Model training complete.

Model Evaluation:
Accuracy: 0.6600

Classification Report:
                        precision    recall  f1-score   support

            ACCOUNTANT       0.67      0.83      0.74        24
              ADVOCATE       0.42      0.54      0.47        24
           AGRICULTURE       0.75      0.46      0.57        13
               APPAREL       0.67      0.32      0.43        19
                  ARTS       0.40      0.38      0.39        21
            AUTOMOBILE       1.00      0.29      0.44         7
              AVIATION       0.80      0.67      0.73        24
               BANKING       0.80      0.70      0.74        23
                   BPO       0.00      0.00      0.00         4
  BUSINESS-DEVELOPMENT       0.51      0.75      0.61        24
                  CHEF       0.80      0.67      0.73        24
          CONSTRUCTION       0.78      0.82      0.80        22
            CONSULTANT       0.33      0.26      0.29   

/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


### Predict Category for a New Resume

To demonstrate the classification capability, let's create a function that takes a raw resume string, applies the same preprocessing steps (cleaning, stop word removal, tokenization), vectorizes it using our trained TF-IDF vectorizer, and then predicts its category using the `svc_model`.

In [ ]:
def predict_resume_category(new_resume_text):
    # 1. Clean the new resume text (using the same clean_text function)
    cleaned_new_resume = clean_text(new_resume_text)

    # 2. Remove stopwords and tokenize (using the same remove_stopwords_and_tokenize function)
    processed_new_resume = remove_stopwords_and_tokenize(cleaned_new_resume)

    # 3. Vectorize the processed resume using the fitted TF-IDF vectorizer
    # We use .transform() here, not .fit_transform(), as the vectorizer is already fitted
    new_resume_vector = tfidf_vectorizer.transform([processed_new_resume])

    # 4. Predict the category using the trained SVC model
    predicted_encoded_category = svc_model.predict(new_resume_vector)

    # 5. Inverse transform the encoded category to get the original category name
    predicted_category_name = le.inverse_transform(predicted_encoded_category)[0]

    return predicted_category_name

# Example usage with a sample resume text (you can replace this with any resume)
sample_resume = "Experienced software engineer with strong skills in Python, Java, and machine learning. Developed scalable web applications and data processing pipelines. Seeking roles in AI/ML engineering."
predicted_category = predict_resume_category(sample_resume)
print(f"The predicted category for the sample resume is: {predicted_category}")

# Let's try with an existing resume from our dataset to see if it predicts correctly
existing_resume_text = df['Resume_str'][100] # Take an arbitrary resume from the dataset
actual_category = df['Category'][100]
predicted_existing_category = predict_resume_category(existing_resume_text)

print(f"\nExisting Resume (ID: {df['ID'][100]}, Actual Category: {actual_category}):\n")
print(f"Predicted Category: {predicted_existing_category}")
print(f"Prediction correct: {actual_category == predicted_existing_category}")

The predicted category for the sample resume is: ENGINEERING

Existing Resume (ID: 10694288, Actual Category: HR):

Predicted Category: HR
Prediction correct: True


### Next Steps for Resume Matching Application

Based on your goal to build a resume matcher and classify resumes, here's a proposed plan:

1.  **Text Preprocessing**: Clean the `Resume_str` column by removing unwanted characters, converting text to lowercase, tokenization, and potentially stemming/lemmatization.
2.  **Feature Engineering (Text Vectorization)**: Convert the processed text into numerical features using techniques like TF-IDF or Word Embeddings (e.g., Word2Vec, FastText).
3.  **Model Selection & Training**: Choose a suitable machine learning model (e.g., Naive Bayes, SVM, Random Forest, or deep learning models for classification) to classify resumes into categories. We can also explore similarity-based models for matching.
4.  **Model Evaluation**: Evaluate the model's performance using appropriate metrics (e.g., accuracy, precision, recall, F1-score).
5.  **Resume Matching Logic**: Develop logic to compare a new resume with existing ones or job descriptions based on their vectorized representations and confidence scores.
6.  **Explainability**: Implement methods to explain *why* a resume is matching, perhaps by highlighting key terms or features that contribute to the match.

Let's start with **Text Preprocessing**.

### Resume Matching Logic: Comparing a Resume to a Job Description

To match a resume to a specific job role, we need to compare the content of the resume with the requirements of the job description. We'll leverage our existing `tfidf_vectorizer` to transform both the resume and the job description into numerical representations. Then, we can calculate the cosine similarity between these two vectors to determine how closely they match.

We'll also aim to provide a basic explanation by highlighting common significant keywords present in both the resume and the job description.

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

def match_resume_to_job(resume_text, job_description_text, top_n_keywords=10):
    # Preprocess and vectorize the resume
    cleaned_resume = clean_text(resume_text)
    processed_resume = remove_stopwords_and_tokenize(cleaned_resume)
    resume_vector = tfidf_vectorizer.transform([processed_resume])

    # Preprocess and vectorize the job description
    cleaned_job_description = clean_text(job_description_text)
    processed_job_description = remove_stopwords_and_tokenize(cleaned_job_description)
    job_description_vector = tfidf_vectorizer.transform([processed_job_description])

    # Calculate cosine similarity
    similarity_score = cosine_similarity(resume_vector, job_description_vector)[0][0]

    # Find common important keywords for explanation
    # Get feature names (words) from the TF-IDF vectorizer
    feature_names = tfidf_vectorizer.get_feature_names_out()

    # Get the TF-IDF scores for the resume and job description
    resume_scores = resume_vector.toarray()[0]
    job_description_scores = job_description_vector.toarray()[0]

    # Identify common keywords with high TF-IDF scores in both
    common_keywords = []
    for i in range(len(feature_names)):
        if resume_scores[i] > 0 and job_description_scores[i] > 0: # Word is present in both
            # Consider words with above-average TF-IDF scores as 'important'
            # A simple heuristic: check if score is above a certain threshold or sort by score
            common_keywords.append((feature_names[i], resume_scores[i] + job_description_scores[i]))

    # Sort common keywords by their combined TF-IDF score (or just resume_scores for simplicity)
    common_keywords = sorted(common_keywords, key=lambda x: x[1], reverse=True)[:top_n_keywords]

    return similarity_score, [word for word, score in common_keywords]

# Define a sample AI Engineer job description
ai_engineer_job_description = """
We are seeking a highly skilled and motivated AI Engineer to join our innovative team. The ideal candidate will have strong expertise in machine learning algorithms, deep learning frameworks (TensorFlow, PyTorch), and data science. Responsibilities include designing, developing, and deploying AI models, conducting research, optimizing model performance, and working with large datasets. Experience with Python, cloud platforms (AWS, GCP, Azure), and MLOps practices is highly desirable. Strong communication and problem-solving skills are essential.
"""

# Example Usage:
# Let's take our sample_resume used earlier
sample_resume_text = "Experienced software engineer with strong skills in Python, Java, and machine learning. Developed scalable web applications and data processing pipelines. Seeking roles in AI/ML engineering."

similarity, keywords = match_resume_to_job(sample_resume_text, ai_engineer_job_description)

print(f"\n--- Resume to AI Engineer Role Match ---")
print(f"Resume: {sample_resume_text[:200]}...")
print(f"\nJob Description: {ai_engineer_job_description[:200]}...")
print(f"\nSimilarity Score: {similarity:.4f}")
print(f"Key matching terms: {', '.join(keywords)}")

# Let's try matching an HR resume to the AI Engineer role
hr_resume_text = df['Resume_str'][100] # From our dataset
similarity_hr, keywords_hr = match_resume_to_job(hr_resume_text, ai_engineer_job_description)

print(f"\n--- HR Resume to AI Engineer Role Match ---")
print(f"HR Resume (ID: {df['ID'][100]}): {hr_resume_text[:200]}...")
print(f"\nSimilarity Score: {similarity_hr:.4f}")
print(f"Key matching terms: {', '.join(keywords_hr)}")



--- Resume to AI Engineer Role Match ---
Resume: Experienced software engineer with strong skills in Python, Java, and machine learning. Developed scalable web applications and data processing pipelines. Seeking roles in AI/ML engineering....

Job Description: 
We are seeking a highly skilled and motivated AI Engineer to join our innovative team. The ideal candidate will have strong expertise in machine learning algorithms, deep learning frameworks (TensorF...

Similarity Score: 0.2932
Key matching terms: python, learning, machine, engineer, seeking, strong, data, skills

--- HR Resume to AI Engineer Role Match ---
HR Resume (ID: 10694288):          HR BENEFITS/LEAVE COORDINATOR       Summary    13 years of Human Resources experience and 27 years of administrative experience working in various settings
*Professional, detail-oriented, exc...

Similarity Score: 0.0157
Key matching terms: include, research, science, working, experience, team, skills


### Creating a Web-Based API for Resume Matching (using Flask)

To move towards a web-based model, we'll create a simple API using the Flask web framework. This API will expose two endpoints:

1.  **`/predict_category`**: Takes raw resume text and returns its predicted category.
2.  **`/match_resume`**: Takes raw resume text and a job description, returning a similarity score and key matching terms.

Since we are in Google Colab, we'll use `pyngrok` to create a publicly accessible URL for our locally running Flask application, allowing you to test it from your browser or other tools.

First, let's install Flask and `pyngrok`.

In [ ]:
# Install Flask and pyngrok (for creating a public URL)
!pip install Flask==2.3.0 pyngrok==6.1.0

# Clear any existing ngrok tunnels if the cell is re-run
from pyngrok import ngrok
ngrok.kill()

print("Flask and pyngrok installed.")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 698.7/698.7 kB 11.7 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.0/97.0 kB 6.9 MB/s eta 0:00:00
  Created wheel for pyngrok: filename=pyngrok-6.1.0-py3-none-any.whl size=20583 sha256=a01437098e92d7f65bfb5e432eb61f8ec5c81f06f585b7c94514551330a680ff
  Stored in directory: /root/.cache/pip/wheels/fb/3f/a3/907e38b0f1971b997c3423a4ca96e4185fc98f608e68ee9fe4
Successfully built pyngrok
  Attempting uninstall: Flask
    Found existing installation: Flask 3.1.3
    Uninstalling Flask-3.1.3:
      Successfully uninstalled Flask-3.1.3
Flask and pyngrok installed.


### Flask Application Code

Below is the code for our Flask application. It defines the two endpoints as discussed.

**Important Note on Confidentiality:**
In this demonstration, all processing happens within your Colab environment. For a production system handling sensitive data like resumes, you would need to implement robust security measures, including:
*   **Secure Data Storage**: Encrypting resumes at rest and in transit.
*   **Access Control**: Limiting who can access the data and the application.
*   **Anonymization/Pseudonymization**: Removing or masking personally identifiable information (PII).
*   **Compliance**: Adhering to regulations like GDPR, CCPA, etc.
*   **API Security**: Using API keys, OAuth, or other authentication mechanisms to protect your endpoints.

For this Colab demonstration, we'll focus on functionality, but always keep these considerations in mind for real-world applications.

In [ ]:
from flask import Flask, request, jsonify
from pyngrok import ngrok
import os

# It's good practice to encapsulate the Flask app creation
app = Flask(__name__)

# Suppress Flask development server output if desired
import logging
log = logging.getLogger('werkzeug')
log.setLevel(logging.ERROR)

# Define the base URL for ngrok
NGROK_TUNNEL = None

@app.route('/')
def home():
    return 'Resume Matcher API is running!'

@app.route('/predict_category', methods=['POST'])
def api_predict_category():
    data = request.get_json()
    if not data or 'resume_text' not in data:
        return jsonify({'error': 'Missing resume_text in request body'}), 400

    resume_text = data['resume_text']
    try:
        predicted_category = predict_resume_category(resume_text)
        return jsonify({'predicted_category': predicted_category}), 200
    except Exception as e:
        return jsonify({'error': str(e)}), 500

@app.route('/match_resume', methods=['POST'])
def api_match_resume():
    data = request.get_json()
    if not data or 'resume_text' not in data or 'job_description_text' not in data:
        return jsonify({'error': 'Missing resume_text or job_description_text in request body'}), 400

    resume_text = data['resume_text']
    job_description_text = data['job_description_text']

    try:
        similarity_score, matching_keywords = match_resume_to_job(resume_text, job_description_text)
        return jsonify({
            'similarity_score': similarity_score,
            'key_matching_terms': matching_keywords
        }), 200
    except Exception as e:
        return jsonify({'error': str(e)}), 500

def start_flask_app():
    global NGROK_TUNNEL
    # Get your authtoken from https://dashboard.ngrok.com/auth/your-authtoken
    # It's recommended to store this in Colab secrets
    # userdata.get('NGROK_AUTH_TOKEN')
    ngrok.set_auth_token('3FALltJuOOeR1fgvOeN6u0Xcsak_A93Nm9c2K7Hkseb1SGVF') # Use the provided authtoken

    # If you have an ngrok authtoken, you can uncomment the line above and provide it.
    # Without an authtoken, ngrok tunnels might be rate-limited or expire sooner.

    port = 5000 # Default Flask port

    # Setup ngrok tunnel
    print("Starting ngrok tunnel...")
    NGROK_TUNNEL = ngrok.connect(port)
    print(f" * ngrok tunnel: {NGROK_TUNNEL.public_url}")
    print(f" * Flask app running on: http://127.0.0.1:{port}/")
    print(" * To stop the app and tunnel, interrupt this cell (Ctrl+C).")

    # Run Flask app
    app.run(port=port, use_reloader=False)

# Call the function to start the app
# Note: This cell will run indefinitely until interrupted.
# To stop, click the 'stop' button on the cell or Restart Runtime.
# Ensure all functions like `clean_text`, `remove_stopwords_and_tokenize`, `tfidf_vectorizer`, `predict_resume_category`, `match_resume_to_job`, and `le` are defined and globally accessible from previous cells.

# Only uncomment and run this if you want to start the web server
start_flask_app()

Starting ngrok tunnel...
 * ngrok tunnel: https://slashed-woof-retorted.ngrok-free.dev
 * Flask app running on: http://127.0.0.1:5000/
 * To stop the app and tunnel, interrupt this cell (Ctrl+C).
 * Serving Flask app '__main__'
 * Debug mode: off


ERROR:root:Unexpected exception finding object shape
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/google/colab/_debugpy_repr.py", line 54, in get_shape
    shape = getattr(obj, 'shape', None)
            ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/werkzeug/local.py", line 318, in __get__
    obj = instance._get_current_object()
          ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/werkzeug/local.py", line 519, in _get_current_object
    raise RuntimeError(unbound_message) from None
RuntimeError: Working outside of request context.

This typically means that you attempted to use functionality that needed
an active HTTP request. Consult the documentation on testing for
information about how to avoid this problem.
ERROR:root:Unexpected exception finding object shape
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/google/colab/_debugpy_repr.py", line 5

### How to Run and Test the API

To run the Flask API, **uncomment the last line (`start_flask_app()`)** in the cell above and execute it. This cell will then run indefinitely, hosting your API.

`pyngrok` will provide a public URL (e.g., `https://xxxx-xx-xxx-xx-xxx.ngrok-free.app`). You can use this URL to send POST requests.

**Example `curl` requests to test the API:**

**1. Predict Resume Category:**

```bash
# Replace <YOUR_NGROK_URL> with the actual URL provided by ngrok
curl -X POST <YOUR_NGROK_URL>/predict_category \
     -H "Content-Type: application/json" \
     -d '{"resume_text": "Highly skilled software developer with expertise in Python, machine learning, and cloud technologies. Proven track record in developing scalable applications."}'
```

**2. Match Resume to Job Description:**

```bash
# Replace <YOUR_NGROK_URL> with the actual URL provided by ngrok
curl -X POST <YOUR_NGROK_URL>/match_resume \
     -H "Content-Type: application/json" \
     -d '{
           "resume_text": "Experienced software engineer with strong skills in Python, Java, and machine learning. Developed scalable web applications and data processing pipelines. Seeking roles in AI/ML engineering.",
           "job_description_text": "We are seeking a highly skilled and motivated AI Engineer with strong expertise in machine learning algorithms, deep learning frameworks (TensorFlow, PyTorch), and data science. Experience with Python and cloud platforms (AWS, GCP, Azure) is highly desirable."
         }'
```

When you are done testing, simply interrupt the cell execution in Colab (click the square stop button next to the cell) to stop the Flask application and close the `ngrok` tunnel.

### Testing the API from Colab

Let's test the `/predict_category` endpoint using a `curl` command executed from a Colab cell. Remember to replace `YOUR_NGROK_URL` if your tunnel URL changes (it's currently `https://slashed-woof-retorted.ngrok-free.dev`).

In [ ]:
# Test the /predict_category endpoint
print("Testing /predict_category endpoint...")
!curl -X POST https://slashed-woof-retorted.ngrok-free.dev/predict_category \
     -H "Content-Type: application/json" \
     -d '{"resume_text": "Experienced machine learning engineer with a strong background in Python, deep learning frameworks like TensorFlow and PyTorch, and cloud platforms. Proven ability to deploy scalable AI solutions."}'

print("\n\nTesting /match_resume endpoint...")
# Test the /match_resume endpoint
!curl -X POST https://slashed-woof-retorted.ngrok-free.dev/match_resume \
     -H "Content-Type: application/json" \
     -d '{
           "resume_text": "Experienced software engineer with strong skills in Python, Java, and machine learning. Developed scalable web applications and data processing pipelines. Seeking roles in AI/ML engineering.",
           "job_description_text": "We are seeking a highly skilled and motivated AI Engineer with strong expertise in machine learning algorithms, deep learning frameworks (TensorFlow, PyTorch), and data science. Experience with Python and cloud platforms (AWS, GCP, Azure) is highly desirable."
         }'
